In [ ]:
from dotenv import load_dotenv
from gensim.models import Word2Vec
from os.path import join, dirname
from sklearn.preprocessing  import OneHotEncoder
from nltk.tokenize          import  RegexpTokenizer
import sys
np.set_printoptions(threshold=sys.maxsize)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import json
# Obtain our default environment
path_to_env = os.path.join('..','.env')
print(load_dotenv(path_to_env))
default_path = os.environ['DEFAULT_PATH']
print(default_path)

True
/home/drew/FL-with-MIMIC


In [2]:
caml_mimic_path  = os.path.join(default_path,'camlmimic')
mimic_data_path  = os.path.join(caml_mimic_path,'mimicdata')
mimic3_data_path = os.path.join(mimic_data_path,'mimic3')

train_50_path = os.path.join(mimic3_data_path,'train_50.csv')
test_50_path  = os.path.join(mimic3_data_path,'test_50.csv')

Load in our train and test dataset

In [3]:
train_50 = pd.read_csv(train_50_path,delimiter=',')
test_50  = pd.read_csv(test_50_path,delimiter=',')
train_50.head(n = 3)

,SUBJECT_ID,HADM_ID,TEXT,LABELS,length
0,7908,182396,admission date discharge date date of birth se...,287.5;584.9;45.13,105
1,11231,183363,admission date discharge date date of birth se...,96.71;272.4;401.9,106
2,3184,144347,admission date discharge date date of birth se...,530.81,117


In [4]:
all_labels = list({l for row in train_50['LABELS'] for l in row.split(';')})
enc = OneHotEncoder().fit(np.array(all_labels).reshape(-1,1)); enc
len(all_labels)

50

Encode our Labels

In [5]:
def encode_onehot_label(row):
    final_label = np.zeros(shape = (1,50)) # (1, label_space)
    for label in row.split(';'):
        label = np.array(label).reshape(1, -1)
        label = enc.transform(label).A
        final_label += label
        # final_label = final_label.flatten()
    return final_label

#Y_train = train_50['LABELS'].apply(func = encode_onehot_label)
Y_train = np.array(list(map(encode_onehot_label, train_50['LABELS'].to_numpy())))
Y_train = Y_train.squeeze(axis = 1)

Y_test = np.array(list(map(encode_onehot_label, test_50['LABELS'].to_numpy())))
Y_test = Y_test.squeeze(axis = 1)

In [9]:
import torch

In [11]:
print(np.sum(Y_train[0]))
Y_train[0], train_50['LABELS'].iloc[0]

3.0


(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),
 '287.5;584.9;45.13')

In [12]:
np.save('Y_test.npy' , Y_test)
np.save('Y_train.npy', Y_train)

----

Load in our CBOW model

In [13]:
model_weights = os.path.join(mimic3_data_path,'processed_full.w2v')
model         = Word2Vec.load(model_weights)

In [17]:
model.wv.most_similar('cancer') # -> seems to perform well

[('ca', 0.7889856100082397),
 ('carcinoma', 0.786292314529419),
 ('caner', 0.752373993396759),
 ('cancers', 0.731344997882843),
 ('adenoca', 0.7046376466751099),
 ('melanoma', 0.6844133734703064),
 ('adenocarcinoma', 0.6756765246391296),
 ('rcc', 0.6593096852302551),
 ('dcis', 0.6501405239105225),
 ('tumors', 0.6480112075805664)]

In [38]:
vocab_size, embed_size = model.wv.vectors.shape
print(vocab_size,embed_size)

150853 100


In [35]:
MIN_TOKEN_LEN = 2500 #max([len(entry) for entry in train_dataset['TEXT']]) # -> 7567
def padding_trunc(row):
    MIN_TOKEN_LEN = 2500
    row = np.array(row)
    # If its smaller we need to truncate
    if len(row) > MIN_TOKEN_LEN:
        return row[:MIN_TOKEN_LEN]
    # Otherwise we pad it with zeroes
    return np.pad(row, pad_width=(0, MIN_TOKEN_LEN - len(row)), mode='constant', constant_values=(vocab_size))

In [39]:
def text_hot_enc(row) -> int:
    result = []
    words = row.split(" ")# Split based on spaces
    for word in words:
        # Access our global word2vec model
        # This also filters out any words that are not in our vocabulary
        if word in model.wv:
            result.append(int(model.wv.key_to_index[word]))
    return padding_trunc(result)

#Y_train = train_50['LABELS'].apply(func = encode_onehot_label)
X_train = np.array(list(map(text_hot_enc, train_50['TEXT'].to_numpy())))
X_test  = np.array(list(map(text_hot_enc, test_50['TEXT'].to_numpy())))

In [45]:
X_train[:2,-5:]

array([[150853, 150853, 150853, 150853, 150853],
       [150853, 150853, 150853, 150853, 150853]])

In [46]:
np.save('X_test.npy' , X_test)
np.save('X_train.npy', X_train)

Truncate or pad each row to be exactly 2500 tokens long

In [ ]:
#current_dir    = os.path.join(default_path,'Replicating Mullenbach')
#train_csv_path = os.path.join(current_dir,'train_50.csv')
#test_csv_path  = os.path.join(current_dir,'test_50.csv')